# Prefect YAML: Hands-On Tutorial

Welcome to the **prefect-yaml** tutorial! This notebook demonstrates:
1. Authoring and executing workflows from YAML without boilerplate Python.
2. Strongly typed input contracts with automatic CLI/string coercion.
3. Building multi-step DAG pipelines with data passing.
4. Incremental execution with deterministic xxHash3 manifest caching.
5. Partitioning and routing items across parallel flows.


## 1. Setup & Imports

In [ ]:
import json
from pathlib import Path
from prefect_yaml import flow_from_yaml, workflow
from prefect_yaml.contracts import InputSpec, validate_workflow_inputs
from prefect_yaml.cache import ManifestCache, compute_step_fingerprint
from prefect_yaml.routing import partition_items, RouterRule

print("prefect-yaml imported successfully!")


## 2. Quickstart: Single-Step Workflow

You can define a workflow inline as a YAML string or load it from a file.
All defaults and inputs are auto-wired to the target function.

In [ ]:
@workflow(name="multiply_numbers", description="Multiply two numbers")
def multiply(a: int, b: int) -> dict:
    return {"product": a * b}

YAML_SINGLE = """
workflows:
  calc:
    description: "Calculate product of two numbers"
    inputs:
      a: {type: int, default: 6}
      b: {type: int, default: 7}
    run: multiply_numbers
"""

# Create and execute the flow
flow = flow_from_yaml(YAML_SINGLE, values={"a": 8, "b": 9})
result = flow()
print("Result:", result)


## 3. Strongly Typed Input Contracts

Inputs define data types, requiredness, defaults, and constraints (`choices`, `minimum`, `maximum`, `regex`).
Strings from CLI overrides (e.g., `--set count=10 --set dry_run=true`) are automatically coerced.

In [ ]:
input_specs = {
    "batch_size": InputSpec(type="int", minimum=1, maximum=50, default=10),
    "mode": InputSpec(type="enum", choices=["fast", "deep"], default="fast"),
    "active": InputSpec(type="bool", default=False),
    "file_path": InputSpec(type="path", default="/tmp/data.csv"),
}

# Simulate raw string overrides from the CLI
raw_cli_inputs = {
    "batch_size": "25",
    "mode": "deep",
    "active": "yes",
}

validated = validate_workflow_inputs(input_specs, raw_cli_inputs)
print("Validated & Coerced inputs:")
for k, v in validated.items():
    print(f"  {k}: {v!r} (type: {type(v).__name__})")


## 4. Multi-Step DAG Pipeline with Dependencies

Chaining steps using `after:` defines dependencies. Independent branches execute in parallel.
Downstream steps can reference outputs using `${steps.<step_id>.result.<field>}`.

In [ ]:
@workflow(name="step_generate")
def generate_data(count: int) -> dict:
    return {"records": [f"rec_{i}" for i in range(count)]}

@workflow(name="step_process")
def process_data(records: list) -> dict:
    return {"total": len(records), "sample": records[:2]}

YAML_DAG = """
workflows:
  sample_dag:
    inputs:
      count: {type: int, default: 5}
    pipeline:
      - id: gen
        run: step_generate
        with:
          count: "${values.count}"
      - id: proc
        run: step_process
        after: [gen]
        with:
          records: "${steps.gen.result.records}"
"""

dag_flow = flow_from_yaml(YAML_DAG, values={"count": 3})
dag_result = dag_flow()
print("DAG Output:", json.dumps(dag_result, indent=2))


## 5. Incremental Execution with xxHash3 Manifest Caching

Steps declaring `cache: manifest` compute an xxHash3 fingerprint of resolved inputs.
If the fingerprint matches an existing manifest entry, the step is skipped.

In [ ]:
step_inputs = {"file": "docs/report.pdf", "format": "markdown"}
fp1 = compute_step_fingerprint("convert_step", step_inputs)
print("xxHash3 Step Fingerprint:", fp1)

# Manifest cache verification
cache = ManifestCache()
cache.record_success("convert_step", fp1, outputs={"result": {"converted": True}})

print("Is cache fresh?", cache.is_fresh("convert_step", fingerprint=fp1))
print("Force re-run bypasses cache:", cache.is_fresh("convert_step", fingerprint=fp1, force=True))


## 6. Partition Router (Pattern Matching)

The partition router classifies collections of items (e.g. files, URLs) using first-match-wins rules.

In [ ]:
rules = [
    RouterRule(match="**/*.pdf", run="pdf_processor", with_={"dpi": 300}),
    RouterRule(match="https://**", run="web_scraper"),
]

items = ["docs/doc1.pdf", "https://example.com", "docs/doc2.pdf", "data.csv"]
buckets = partition_items(items, rules, default="default_processor")

print("Partitioned Buckets:")
for (target, params_json), data in buckets.items():
    print(f"  Target: {target} | Params: {params_json} | Items: {data['items']}")
